# NB02 · Semana 4 — Extracción de Open-Meteo hacia bronze (Entrega 1)
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

El pipeline de ingesta vive en `scripts/ingesta.py` (lo usan este notebook **y** Airflow: una sola implementación).
Acá lo recorremos paso a paso y hacemos la **demo de E1**: correr dos veces y mostrar que la segunda no agrega nada.

**Dos APIs de Open-Meteo**
| Rango | API | Por qué |
|---|---|---|
| 2022-01-01 → 2025-12-31 | Archive (reanálisis ERA5) | Histórico de máxima calidad |
| 2026-01-01 → ayer | Historical Forecast | Tiene el día anterior completo (ERA5 se publica con días de demora) |

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
from scripts import ingesta
from datetime import date, timedelta

## 1. Watermark
La máxima `fecha` que ya está en bronze. Se **deriva de los datos** (no de un archivo aparte que se pueda desincronizar)
y se usa como **parámetro de la consulta**: `start_date = watermark + 1`. Si la tabla no existe devuelve `None` (primera corrida);
cualquier otro error (credenciales, red) se propaga, para no confundir "falla" con "tabla vacía" y recargar todo.

In [ ]:
print('watermark actual:', ingesta.leer_watermark())

## 2. Extracción con manejo de errores
`extraer(desde, hasta)` parte el rango por fuente y por año, y por cada pedido:
- **verifica el status antes de parsear**;
- **transitorias** (timeout, sin conexión, HTTP 429, 5xx) → hasta 3 intentos con espera 2 s / 4 s;
- **permanentes** (HTTP 400/401/403/404, cuerpo que no es JSON, falta `daily`, columnas de distinto largo, fechas fuera de rango, unidad distinta de °C) → error inmediato, sin reintentar;
- días con **todas** las mediciones nulas (aún no publicados) no se escriben: el watermark no avanza y se vuelven a pedir.

Probamos con una semana (no escribe nada):

In [ ]:
muestra = ingesta.extraer(date(2025, 12, 29), date(2026, 1, 3))
muestra[["fecha", "fuente", "temperature_2m_min", "temperature_2m_max", "precipitation_sum", "weather_code"]]

Un pedido inválido es **permanente** y no se reintenta:

In [ ]:
try:
    ingesta._pedir(ingesta.URL_ARCHIVE, {"latitude": -33.78, "longitude": 151.23, "daily": "variable_inexistente",
                                         "start_date": "2025-01-01", "end_date": "2025-01-02"})
except ingesta.ErrorPermanente as e:
    print("ErrorPermanente ->", e)

## 3. Escritura en bronze (Delta Lake en MinIO)
- Solo fechas nuevas → `append`.
- Si el intervalo incluye fechas ya cargadas (backfill / reproceso) → `MERGE` por `fecha` (upsert).

Bronze guarda el dato **crudo** tal como vino (nombres de la API, sin limpiar) + `fuente` e `ingestado_en` para auditoría.

## 4. Demo E1: correr dos veces

In [ ]:
from deltalake import DeltaTable
def resumen():
    dt = DeltaTable(entorno.BRONZE, storage_options=entorno.STORAGE_OPTIONS)
    df = dt.to_pandas(columns=["fecha", "fuente"])
    return {"version_delta": dt.version(), "filas": len(df), "fechas_unicas": df.fecha.nunique(),
            "desde": df.fecha.min(), "hasta": df.fecha.max()}

print("Corrida 1 ->", ingesta.ingerir(), "filas nuevas"); antes = resumen(); print(antes)

In [ ]:
print("Corrida 2 ->", ingesta.ingerir(), "filas nuevas"); despues = resumen(); print(despues)
assert despues["filas"] == antes["filas"] == despues["fechas_unicas"], "¡duplicados!"
print("✔ Idempotente: la segunda corrida no consultó la API porque no había nada nuevo que pedir")

**Por qué es idempotente (y no es deduplicación):** la segunda corrida lee el watermark, calcula
`desde = watermark + 1 > hasta` y **no le pide nada al origen**. No traemos lo que ya tenemos.

## 5. Reproceso de un período (lo que usa el backfill de E3)
Pedimos explícitamente una semana ya cargada: se vuelve a extraer y se hace MERGE. Las filas no cambian.

In [ ]:
n = ingesta.ingerir(inicio=date(2026, 3, 1), fin=date(2026, 3, 8))
print("reprocesadas:", n); print(resumen())

In [ ]:
# Historial Delta: cada escritura es una versión (auditoría + time travel, ver NB03)
dt = DeltaTable(entorno.BRONZE, storage_options=entorno.STORAGE_OPTIONS)
import pandas as pd
pd.DataFrame(dt.history())[["version", "timestamp", "operation", "operationMetrics"]].head(10)